# Pré-processamento das Bases da SSP-SP
O objetivo é consolidar as fontes obtidas pela transparência da **Secretária de Segurança Pública de São Paulo (SPP-SP)** em um único **dataset**. A unidade de tempo utilizada na consolidação da base será DIA-MES-ANO.

## 1. Bibliotecas e Configuração Inicial

In [1]:
import pandas as pd
import numpy as np
import math

In [2]:
np.random.seed(42)

## 2. Leitura da base consolidada
*datasets/SPDadosCriminais_2022_2026.csv*

In [3]:
df_raw = pd.read_csv('datasets/SPDadosCriminais_2022_2026.csv')

C:\Users\Usuario\AppData\Local\Temp\ipykernel_11096\1190081205.py:1: DtypeWarning: Columns (3,6,9,13,14,15,16,17,18,19,31,32,33,34,35,36,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('datasets/SPDadosCriminais_2022_2026.csv')


***Agregação***
<p>É necessário gerar a quantidade de ocorrências por tipo de crime, localização, dia, mês e ano para subsidiar o cálculo do risco de <b>criminalidade</b>.</p>

## 3. Padronização da Coluna de Cidade

A base possui duas colunas que podem indicar o município do B.O.: `NOME_MUNICIPIO_CIRCUNCRIÇÃO` (município da circunscrição/delegacia responsável [local onde ocorreu o fato]) e `NOME_MUNICIPIO` (município registrado). Como a primeira apresenta uma quantidade relevante de nulos, é necessário avaliar se a segunda pode servir de **fallback**, e então consolidar as duas em uma única coluna `city`.

In [4]:
len(df_raw[df_raw['NOME_MUNICIPIO_CIRCUNCRIÇÃO'].isnull()]) / len(df_raw)

0.10384811135788

In [5]:
city_null = df_raw[df_raw['NOME_MUNICIPIO_CIRCUNCRIÇÃO'].isnull()]

In [6]:
len(city_null[city_null['NOME_MUNICIPIO'].isnull()]) / len(city_null)

0.0

Confirmada a viabilidade do fallback, a coluna `city` é criada a partir de `NOME_MUNICIPIO_CIRCUNCRIÇÃO`, preenchendo os valores nulos com `NOME_MUNICIPIO`.

In [7]:
df_raw['city'] = df_raw['NOME_MUNICIPIO_CIRCUNCRIÇÃO'].fillna(df_raw['NOME_MUNICIPIO'])

In [8]:
df_raw['city'] = df_raw['city'].str.strip()

In [9]:
len(df_raw[df_raw['city'].isnull()]) / len(df_raw)

0.0

## 4. Filtragem por Tipo de Crime

O estudo tem como foco **Roubos e Furtos**. Antes de aplicar qualquer filtro, é feito um levantamento dos valores únicos da coluna `NATUREZA_APURADA`, para identificar todas as categorias existentes e decidir quais devem ser mantidas.

In [10]:
df_raw['NATUREZA_APURADA'].unique()

array(['ESTUPRO', 'FURTO - OUTROS', 'FURTO DE VEÍCULO',
       'HOMICÍDIO DOLOSO',
       'LESÃO CORPORAL CULPOSA POR ACIDENTE DE TRÂNSITO',
       'LESÃO CORPORAL DOLOSA', 'ROUBO - OUTROS', 'ROUBO DE CARGA',
       'ROUBO DE VEÍCULO', 'TENTATIVA DE HOMICIDIO',
       'TRAFICO DE ENTORPECENTES', 'LESÃO CORPORAL CULPOSA – OUTRAS',
       'ESTUPRO DE VULNERÁVEL', 'FURTO DE CARGA',
       'HOMICIDIO CULPOSO POR ACIDENTE DE TRANSITO',
       'HOMICIDIO CULPOSO OUTROS', 'EXTORSÃO MEDIANTE SEQUESTRO',
       'LATROCÍNIO', 'ROUBO A BANCO', 'LESÃO CORPORAL SEGUIDA DE MORTE',
       'HOMICÍDIO DOLOSO POR ACIDENTE DE TRÂNSITO',
       'HOMICÍDIO CULPOSO POR ACIDENTE DE TRÂNSITO',
       'LESÃO CORPORAL CULPOSA - OUTRAS', 'PORTE DE ARMA',
       'PORTE DE ENTORPECENTES', 'TRÁFICO DE ENTORPECENTES',
       'APREENSÃO DE ENTORPECENTES', 'TENTATIVA DE HOMICÍDIO',
       'HOMICÍDIO CULPOSO OUTROS', 'Furto de Veículo', 'Roubo de Veículo',
       'Furto - Outros', 'Estupro', 'Estupro de Vulnerável',
  

Como o objetivo do estudo é analisar **Roubos e Furtos**, serão mantidas as seguintes categorias:
- 'FURTO - OUTROS'
- 'FURTO DE VEÍCULO'
- 'FURTO DE CARGA'
- 'ROUBO - OUTROS'
- 'ROUBO DE CARGA'
- 'ROUBO DE VEÍCULO'
- 'ROUBO A BANCO'
<p>Em seguida, <b>'ROUBO A BANCO'</b> será incorporado em <b>'ROUBO - OUTROS'</b></p>

In [11]:
list_crimes = ['FURTO - OUTROS', 'FURTO DE VEÍCULO', 'ROUBO - OUTROS',
               'ROUBO DE CARGA', 'ROUBO DE VEÍCULO', 'FURTO DE CARGA',
                'ROUBO A BANCO', 'Furto de Veículo', 'Roubo de Veículo',
                    'Furto - Outros']

In [12]:
df_raw = df_raw[df_raw['NATUREZA_APURADA'].isin(list_crimes)]

In [13]:
df_raw['NATUREZA_APURADA'] = df_raw['NATUREZA_APURADA'].str.upper()

In [14]:
df_raw['NATUREZA_APURADA'] = df_raw['NATUREZA_APURADA'].map(lambda x: 'ROUBO - OUTROS' if x == 'ROUBO A BANCO' else x)

In [15]:
df_raw['NATUREZA_APURADA'].unique()

array(['FURTO - OUTROS', 'FURTO DE VEÍCULO', 'ROUBO - OUTROS',
       'ROUBO DE CARGA', 'ROUBO DE VEÍCULO', 'FURTO DE CARGA'],
      dtype=object)

Será mantido apenas B.O. de munícipios de Região Metropolitana de São Paulo. A lista de munícipios está disponível em: https://pt.wikipedia.org/wiki/Regi%C3%A3o_Metropolitana_de_S%C3%A3o_Paulo

## 5. Filtragem Geográfica: Região Metropolitana de São Paulo

O escopo do estudo é restrito aos municípios da **Região Metropolitana de São Paulo (RMSP)**. A lista de municípios é definida manualmente com base na composição oficial da RMSP, e utilizada para filtrar a base pela coluna `city`.

In [16]:
list_city = ['ARUJA', 'BARUERI', 'BIRITIBA-MIRIM', 'CAIEIRAS', 'CAJAMAR',
             'CARAPICUIBA', 'COTIA', 'DIADEMA', 'EMBU DAS ARTES', 'EMBU-GUACU',
                'FERRAZ DE VASCONCELOS', 'FRANCISCO MORATO', 'FRANCO DA ROCHA', 'GUARAREMA',
                    'GUARULHOS', 'ITAPECERICA DA SERRA', 'ITAPEVI', 'ITAQUAQUECETUBA',
                        'JANDIRA', 'JUQUITIBA', 'MAIRIPORA', 'MAUA', 'MOGI DAS CRUZES', 'OSASCO',
                        'PIRAPORA BOM JESUS', 'POA', 'RIBEIRAO PIRES', 'RIO GRANDE DA SERRA',
                            'SALESOPOLIS', 'SANTANA DE PARNAIBA', 'S.ANDRE', 'S.BERNARDO DO CAMPO', 'S.CAETANO DO SUL', 'S.LOURENCO DA SERRA', 'S.PAULO', 'SUZANO', 'TABOAO DA SERRA', 'VARGEM GRANDE PAULISTA']

In [17]:
df_raw = df_raw[df_raw['city'].isin(list_city)]

## 6. Tratamento das Datas de Ocorrência

Antes de derivar mês, dia e ano do B.O., é avaliada a completude da coluna `DATA_OCORRENCIA_BO` e `DATA_REGISTRO`.

In [18]:
len(df_raw[df_raw['DATA_OCORRENCIA_BO'].isnull()]) / len(df_raw)

0.00016727296723061187

In [19]:
len(df_raw[df_raw['DATA_REGISTRO'].isnull()]) / len(df_raw)

0.23576856173638547

In [20]:
len(df_raw[df_raw['DATA_OCORRENCIA_BO'].isnull()])

436

In [21]:
len(df_raw[df_raw['DATA_REGISTRO'].isnull()])

614535

Devido ao baixo volume, registros sem `DATA_OCORRENCIA_BO` serão excluídos.

In [22]:
df_raw = df_raw[df_raw['DATA_OCORRENCIA_BO'].notnull()]

Notou-se que a coluna `ANO_BO` representa o ano de **registro do Boletim de Ocorrência**, e não necessariamente o ano em que o fato ocorreu. Por isso, para garantir consistência na análise temporal, o `ANO` também será extraído a partir da coluna `DATA_OCORRENCIA_BO`, junto com `MES` e `DIA`.

`DATA_OCORRENCIA_BO` é convertids para o tipo `datetime`, com `errors='coerce'` transformando qualquer valor inválido em nulo (`NaT`), em vez de interromper a execução.

In [23]:
df_raw['DATA_OCORRENCIA_BO'] = pd.to_datetime(df_raw['DATA_OCORRENCIA_BO'], errors='coerce')

As colunas `ANO_BO`, `MES_BO` e `DIA_BO` são extraídas de `DATA_OCORRENCIA_BO`.

In [24]:
df_raw['ANO_BO'] = df_raw['DATA_OCORRENCIA_BO'].dt.year.astype('Int64')
df_raw['MES_BO'] = df_raw['DATA_OCORRENCIA_BO'].dt.month.astype('Int64')
df_raw['DIA_BO'] = df_raw['DATA_OCORRENCIA_BO'].dt.day.astype('Int64')

**Filtro aplicado:** registros com `ANO_BO <= 2021` foram descartados como ruído. Sua incidência é baixa, inferior à taxa de dados ausentes em `DATA_REGISTRO`.

In [25]:
len(df_raw[df_raw['ANO_BO'] <= 2021]) / len(df_raw)

0.005639116497485498

In [26]:
df_raw = df_raw[df_raw['ANO_BO'] > 2021]

In [27]:
df_raw = df_raw[~((df_raw['ANO_BO'] == 2026) & (df_raw['MES_BO'] == 7))]

In [28]:
len(df_raw[df_raw['MES_BO'].isnull() | df_raw['DIA_BO'].isnull() | df_raw['ANO_BO'].isnull()])

0

## 7. Tratamento de Registros Duplicados
Será avaliado a presença de registros duplicados e seus impactos

In [29]:
df_raw.duplicated(subset=['LATITUDE', 'LONGITUDE', 'ANO_BO', 'MES_BO', 'DIA_BO', 'city', 'NATUREZA_APURADA', 'NUM_BO']).sum()

93

In [30]:
df_raw.duplicated(subset=['LATITUDE', 'LONGITUDE', 'ANO_BO', 'MES_BO', 'DIA_BO', 'city', 'NATUREZA_APURADA', 'NUM_BO']).sum() / len(df_raw) * 100

0.0035888712575404885

Cerca de **0,004%** da base são **registros duplicados**. Portanto, como um dado duplicado não agrega informação, logo serão removidos da base.

In [31]:
df_raw = df_raw.drop_duplicates(subset=['LATITUDE', 'LONGITUDE', 'ANO_BO', 'MES_BO', 'DIA_BO', 'city', 'NATUREZA_APURADA', 'NUM_BO'], keep='first')

## 8. Tratamento de Coordenadas Geográficas

Antes de qualquer correção, é avaliada a proporção de valores nulos em `LATITUDE` e `LONGITUDE`, individualmente e em conjunto, para dimensionar o problema e orientar a estratégia de imputação.

In [32]:
len(df_raw[df_raw['LATITUDE'].isnull()]) / len(df_raw)

0.1131604001310564

In [33]:
len(df_raw[df_raw['LONGITUDE'].isnull()]) / len(df_raw)

0.1131604001310564

In [34]:
len(df_raw[df_raw['LATITUDE'].isnull() | df_raw['LONGITUDE'].isnull()])

293227

### 8.1 Identificando Coordenadas Inválidas

É criada a coluna `COORD_IMPUTADA`, que sinaliza registros cuja coordenada precisa ser tratada/imputada. Um registro é marcado como inválido quando:

- `LATITUDE` ou `LONGITUDE` são nulos; **ou**
- o valor (como string) não começa com `-` (coordenadas no Brasil devem ser negativas); **ou**
- o valor é literalmente o caractere `'-'` (sem dígitos).

In [35]:
df_raw['COORD_IMPUTADA'] = df_raw['LATITUDE'].isnull() | df_raw['LONGITUDE'].isnull() | (~df_raw['LATITUDE'].astype(str).str.match(r'^-')) | (~df_raw['LONGITUDE'].astype(str).str.match(r'^-')) | (df_raw['LATITUDE'] == '-') | (df_raw['LONGITUDE'] == '-')

### 8.2 Funções de Correção de Coordenadas

São definidas três funções auxiliares:

- **`fix_coord_lat` / `fix_coord_long`**: zeram a coordenada de registros marcados como `COORD_IMPUTADA`, preparando-os para receberem o valor definitivo mais adiante;
- **`fix_coord`**: corrige erros de **escala** — coordenadas gravadas sem o ponto decimal (ex: `-4664112.0` em vez de `-46.64112`) — dividindo o valor por 10 repetidamente até que seu módulo fique dentro do limite geográfico válido (90° para latitude, 180° para longitude).

In [36]:
def fix_coord_lat(row):
    return 0 if row['COORD_IMPUTADA'] else row['LATITUDE']

def fix_coord_long(row):
    return 0 if row['COORD_IMPUTADA'] else row['LONGITUDE']

def fix_coord(value, limit=90):
    fixed_value = value
    while abs(fixed_value) > limit:
        fixed_value /= 10
    return fixed_value

Substitui valores **inválidos** por *0*

In [37]:
df_raw['LONGITUDE'] = df_raw.apply(fix_coord_long, axis=1)
df_raw['LATITUDE'] = df_raw.apply(fix_coord_lat, axis=1)

In [38]:
df_raw['LONGITUDE'] = df_raw['LONGITUDE'].astype('Float64')
df_raw['LATITUDE'] = df_raw['LATITUDE'].astype('Float64')

**Corrigi** as coordenadas para um **intervalo válido**

In [39]:
df_raw['LONGITUDE'] = df_raw['LONGITUDE'].map(fix_coord)
df_raw['LATITUDE'] = df_raw['LATITUDE'].map(fix_coord)

In [40]:
df_raw['LONGITUDE'] = df_raw['LONGITUDE'].replace(0, np.nan)
df_raw['LATITUDE'] = df_raw['LATITUDE'].replace(0, np.nan)

### 8.3 Imputação por Centróide da Cidade

Para os registros sem coordenada válida, a estratégia adotada é preencher com a **mediana** de latitude/longitude dos demais registros da mesma cidade (`city`). A mediana é usada em vez da média por ser mais robusta a eventuais outliers remanescentes na base.

In [41]:
median_point_city = (
    df_raw.dropna(subset=['LATITUDE', 'LONGITUDE'])
    .groupby('city')[['LATITUDE', 'LONGITUDE']]
    .median()
)

In [42]:
df_raw['LATITUDE'] = df_raw['LATITUDE'].fillna(df_raw['city'].map(median_point_city['LATITUDE']))
df_raw['LONGITUDE'] = df_raw['LONGITUDE'].fillna(df_raw['city'].map(median_point_city['LONGITUDE']))

## 9. Junção de Categorias

A coluna `NATUREZA_APURADA` apresenta poucas variações, e nenhuma outra coluna da base permite identificar o tipo específico de delito. Por isso, o trabalho adotará categorias mais genéricas, agrupando as ocorrências em **FURTO** e **ROUBO**.

In [43]:
list_furto = ['FURTO - OUTROS', 'FURTO DE VEÍCULO', 'FURTO DE CARGA']

In [44]:
df_new = df_raw.copy()

In [45]:
df_new['NATUREZA_APURADA'] = df_raw['NATUREZA_APURADA'].map(lambda x: 'FURTO' if x in list_furto else 'ROUBO')

In [46]:
df_new['NATUREZA_APURADA'].unique()

array(['FURTO', 'ROUBO'], dtype=object)

## 10. Agregação Final

Com a base limpa e padronizada, o último passo é agregá-la por **latitude, longitude e data (DIA-MES-ANO)**, gerando as métricas que servirão de base para o cálculo do risco de criminalidade:

- `cnt_total`: contagem **total** de ocorrências por combinação de latitude/longitude/data;
- `cnt_by_type`: contagem de ocorrências **por tipo de crime** (`NATUREZA_APURADA`), uma coluna por categoria, para a mesma combinação de chaves.

In [47]:
cnt_total = (
    df_new.groupby(['LATITUDE', 'LONGITUDE', 'DIA_BO', 'city', 'MES_BO', 'ANO_BO'])
        .size()
        .reset_index(name='cnt_crime')
)

In [48]:
cnt_by_type = (
    df_new.pivot_table(
        index=['LATITUDE', 'LONGITUDE', 'DIA_BO', 'city', 'MES_BO', 'ANO_BO'],
        columns='NATUREZA_APURADA',
        aggfunc='size',
        fill_value=0
    ).reset_index()
)

As duas agregações (`cnt_total` e `cnt_by_type`) são unidas em um único dataframe final (`df_group`), contendo a contagem total e a contagem por tipo de crime para cada combinação de localização e data.

In [49]:
df_group = cnt_total.merge(cnt_by_type, on=['LATITUDE', 'LONGITUDE', 'DIA_BO', 'city', 'MES_BO', 'ANO_BO'])

In [50]:
df_group

,LATITUDE,LONGITUDE,DIA_BO,city,MES_BO,ANO_BO,cnt_crime,FURTO,ROUBO
0,-28.304600,-54.271865,10,GUARULHOS,12,2024,1,1,0
1,-24.972870,-48.402852,16,TABOAO DA SERRA,10,2025,1,0,1
2,-24.972870,-48.402852,27,EMBU DAS ARTES,11,2025,1,0,1
3,-24.969331,-48.361020,28,ITAPECERICA DA SERRA,2,2024,1,0,1
4,-24.969324,-48.361338,9,JUQUITIBA,7,2023,1,0,1
...,...,...,...,...,...,...,...,...,...
2074437,-21.039556,-49.930865,8,S.PAULO,2,2023,1,1,0
2074438,-20.325371,-48.310855,8,S.PAULO,6,2026,1,1,0
2074439,-19.882014,-43.896489,3,S.PAULO,4,2025,1,0,1
2074440,-15.477469,-44.371940,19,S.PAULO,7,2025,1,0,1


In [52]:
df_group.to_csv('datasets/database_processed_by_day.csv', index=False)